# Project 5 — From Static to Contextual (UD English LinES)
End-to-end, self-contained: **Runtime → Change runtime type → T4 GPU**, then **Runtime → Run all**.

| Task | What runs | Approx. time on T4 |
|---|---|---|
| 1 | PPMI+SVD vectors, WordSim-353, Google analogy, 20-sentence MST check | < 1 min |
| 2 | BERT feature extraction (13 layers, cached) + 72 structural probes (12 layers × 3 ranks × 2 probes × 30 epochs) | ~15–40 min |
| 3 | Layer-ℓ* pooled static vectors, same evaluations, significance tests, layer sweep | ~2 min |

Results: `outputs/REPORT.md` (rendered at the end), plots in `outputs/task*/`, probe checkpoints in `checkpoints/`.
Probe checkpoints are reused if the runtime restarts (`task2.resume: true`); mount Drive (optional cell) to survive disconnects.

In [ ]:
# 0. GPU check
!nvidia-smi --query-gpu=name,memory.total,memory.used,driver_version --format=csv || echo "No GPU - will run on CPU (slow)"
import torch
print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# 1. Dependencies (Colab already has torch/transformers/datasets; this only fills gaps)
!pip -q install "transformers>=4.44,<6" "scikit-learn>=1.3" "pandas>=2.2" pyyaml scipy matplotlib

In [ ]:
# 2. (Optional) persist everything on Google Drive so a disconnect does not lose probe checkpoints
USE_DRIVE = False
import os
if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT = '/content/drive/MyDrive/nlp_project5'
else:
    PROJECT = '/content/project5'
os.makedirs(PROJECT + '/src/p5', exist_ok=True)
%cd {PROJECT}

## 3. Project source files
(Generated from the repository — identical to the Docker build.)

In [ ]:
%%writefile config.yaml
# Project 5 - UD English LinES. All paths are relative to this file.
seed: 42
device: auto            # auto | cuda | cpu

paths:
  data_dir: dataset          # downloaded CoNLL-U + benchmark files (originals never modified)
  output_dir: outputs        # results, plots, REPORT.md, run.log
  checkpoint_dir: checkpoints
  cache_dir: cache           # cached BERT word features (~2 GB, all 13 layers, 3 splits)

data:
  source: auto               # auto (HF -> HF parquet -> UD GitHub) | hf | github
  hf_name: universal_dependencies
  hf_config: en_lines
  # UD v2.7 = the release the HF `universal_dependencies` dataset is built from
  github_url: https://raw.githubusercontent.com/UniversalDependencies/UD_English-LinES/r2.7/en_lines-ud-{split}.conllu

benchmarks:
  wordsim_path: null         # set to a local file to override download
  analogy_path: null
  wordsim_urls:
    - https://raw.githubusercontent.com/piskvorky/gensim/develop/gensim/test/test_data/wordsim353.tsv
  analogy_urls:
    - https://raw.githubusercontent.com/piskvorky/gensim/develop/gensim/test/test_data/questions-words.txt
    - https://raw.githubusercontent.com/tmikolov/word2vec/master/questions-words.txt

structural_check:
  n_sentences: 20
  seed: 42

task1:
  window: 5                  # symmetric, within sentence
  alpha: 1.0                 # 1.0 = plain PPMI (as specified)
  dim: 100
  svd_power: 1.0             # vectors = U * Sigma^power
  ablation_alpha: [1.0, 0.75]
  ablation_svd_power: [1.0, 0.5]

bert:
  model_name: bert-base-uncased
  extract_batch_size: auto   # auto-sized from free VRAM
  mixed_precision: true      # bf16 on Ampere+, fp16 otherwise (inference only)

task2:
  layers: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
  ranks: [32, 64, 128]
  lr: 0.001
  batch_size: 20
  epochs: 30
  select_on: dev             # choose l* on dev UUAS (avoids test leakage); 'test' = literal brief
  resume: true               # reuse saved probe checkpoints if present

task3:
  layer: null                # null -> read l* from Task 2
  n_bootstrap: 2000
  sweep_include_layer0: true


In [ ]:
%%writefile run_all.py
#!/usr/bin/env python3
"""One entry point for the whole project: Task 1 -> Task 2 -> Task 3 -> REPORT.md.

  python run_all.py                       # everything, config.yaml
  python run_all.py --tasks 2 3           # subset (Task 3 needs Task 1 + Task 2 outputs)
  python run_all.py --config config.smoke.yaml
"""
import argparse
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path(__file__).resolve().parent / "src"))

from p5 import report, task1_ppmi, task2_probe, task3_bert_static  # noqa: E402
from p5.utils import LOG, load_config, setup_logging  # noqa: E402


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--config", default="config.yaml")
    ap.add_argument("--tasks", nargs="+", type=int, default=[1, 2, 3], choices=[1, 2, 3])
    ap.add_argument("--device", choices=["auto", "cuda", "cpu"], default=None)
    args = ap.parse_args()
    overrides = {"device": args.device} if args.device else {}
    cfg = load_config(args.config, overrides)
    setup_logging(Path(cfg["paths"]["output_dir"]) / "run.log")
    LOG.info("Config: %s | tasks %s", Path(args.config).resolve(), args.tasks)
    steps = {1: task1_ppmi.run, 2: task2_probe.run, 3: task3_bert_static.run}
    for t in sorted(args.tasks):
        t0 = time.time()
        LOG.info("=" * 25 + f" TASK {t} " + "=" * 25)
        steps[t](cfg)
        LOG.info("Task %d finished in %.1f min", t, (time.time() - t0) / 60)
    report.build(cfg)
    LOG.info("All done. Results in %s", cfg["paths"]["output_dir"])
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile src/p5/__init__.py


In [ ]:
%%writefile src/p5/utils.py
"""Shared utilities: config, seeding, device detection, logging, IO."""
from __future__ import annotations

import json
import logging
import os
import random
import sys
from pathlib import Path
from typing import Any

import numpy as np
import yaml

LOG = logging.getLogger("p5")


def setup_logging(log_file: Path | None = None) -> None:
    fmt = logging.Formatter("%(asctime)s | %(levelname)-7s | %(message)s", "%H:%M:%S")
    root = logging.getLogger("p5")
    root.setLevel(logging.INFO)
    root.handlers.clear()
    sh = logging.StreamHandler(sys.stdout)
    sh.setFormatter(fmt)
    root.addHandler(sh)
    if log_file is not None:
        log_file.parent.mkdir(parents=True, exist_ok=True)
        fh = logging.FileHandler(log_file, mode="a", encoding="utf-8")
        fh.setFormatter(fmt)
        root.addHandler(fh)
    root.propagate = False


def load_config(path: str | os.PathLike, overrides: dict | None = None) -> dict:
    path = Path(path)
    if not path.is_file():
        raise FileNotFoundError(f"Config file not found: {path.resolve()}")
    with open(path, encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    if not isinstance(cfg, dict):
        raise ValueError(f"Config {path} did not parse to a mapping")
    for k, v in (overrides or {}).items():
        _set_nested(cfg, k, v)
    # Resolve all paths relative to the project root (directory of the config file)
    root = path.resolve().parent
    for key in ("data_dir", "output_dir", "checkpoint_dir", "cache_dir"):
        p = Path(cfg["paths"][key])
        cfg["paths"][key] = str(p if p.is_absolute() else (root / p))
    return cfg


def _set_nested(cfg: dict, dotted: str, value: Any) -> None:
    keys = dotted.split(".")
    d = cfg
    for k in keys[:-1]:
        d = d.setdefault(k, {})
    d[keys[-1]] = value


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    try:
        import torch

        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except ImportError:
        pass


def get_device(prefer: str = "auto"):
    """Pick the compute device and log GPU model / VRAM. Falls back to CPU."""
    import torch

    if prefer != "cpu" and torch.cuda.is_available():
        dev = torch.device("cuda:0")
        props = torch.cuda.get_device_properties(dev)
        free, total = torch.cuda.mem_get_info(dev)
        LOG.info(
            "GPU: %s | VRAM total %.1f GB, free %.1f GB | CUDA %s | bf16 %s",
            props.name, total / 2**30, free / 2**30, torch.version.cuda,
            torch.cuda.is_bf16_supported(),
        )
        return dev
    if prefer == "cuda":
        LOG.warning("CUDA requested but not available - falling back to CPU")
    LOG.warning("Running on CPU. BERT extraction and probe training will be slow.")
    return torch.device("cpu")


def gpu_mem_gb(device) -> float:
    import torch

    if device.type != "cuda":
        return 0.0
    return torch.cuda.mem_get_info(device)[1] / 2**30


def amp_dtype(device):
    """Mixed-precision dtype for inference: bf16 on Ampere+, fp16 otherwise; None on CPU."""
    import torch

    if device.type != "cuda":
        return None
    return torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16


def save_json(obj: Any, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, default=_json_default)


def load_json(path: Path) -> Any:
    if not Path(path).is_file():
        raise FileNotFoundError(f"Expected file is missing: {path}. Run the earlier task first.")
    with open(path, encoding="utf-8") as f:
        return json.load(f)


def _json_default(o):
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, np.ndarray):
        return o.tolist()
    if isinstance(o, Path):
        return str(o)
    raise TypeError(f"Not JSON serialisable: {type(o)}")


def ensure_dirs(cfg: dict) -> dict[str, Path]:
    p = {k: Path(v) for k, v in cfg["paths"].items()}
    for v in p.values():
        v.mkdir(parents=True, exist_ok=True)
    return p


In [ ]:
%%writefile src/p5/data.py
"""UD English LinES loading, validation, statistics and dependency-tree utilities.

Loading order (first that works wins, the source is logged and saved):
  1. HuggingFace `load_dataset("universal_dependencies", "en_lines")` (as the brief prescribes).
     Script-based datasets were removed in `datasets>=4`, so this often fails on new installs.
  2. The parquet conversion of the same HF dataset (`revision="refs/convert/parquet"`).
  3. The official UD GitHub release r2.7 CoNLL-U files - the exact release the HF dataset wraps.
All three give the same tokens/head/deprel fields; nothing else is used.
"""
from __future__ import annotations

import collections
import urllib.request
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np

from .utils import LOG, save_json

SPLITS = ("train", "dev", "test")
HF_SPLIT = {"train": "train", "dev": "validation", "test": "test"}


@dataclass
class Sentence:
    sid: str
    tokens: list[str]
    heads: list[int]          # 1-based head index per token, 0 = root
    deprels: list[str]
    _dist: np.ndarray | None = field(default=None, repr=False)
    _depth: np.ndarray | None = field(default=None, repr=False)

    def __len__(self) -> int:
        return len(self.tokens)

    @property
    def punct_mask(self) -> np.ndarray:
        """True for punctuation tokens (deprel == punct), excluded from UUAS as in Hewitt & Manning."""
        return np.array([d.split(":")[0] == "punct" for d in self.deprels], dtype=bool)

    @property
    def distances(self) -> np.ndarray:
        if self._dist is None:
            self._dist = tree_distances(self.heads)
        return self._dist

    @property
    def depths(self) -> np.ndarray:
        if self._depth is None:
            root = self.heads.index(0)
            self._depth = self.distances[root].copy()
        return self._depth


# --------------------------------------------------------------------------------------------
# Tree utilities
# --------------------------------------------------------------------------------------------
def tree_distances(heads: list[int]) -> np.ndarray:
    """All-pairs path length in the undirected dependency tree (BFS from every node)."""
    n = len(heads)
    adj: list[list[int]] = [[] for _ in range(n)]
    for i, h in enumerate(heads):
        if h > 0:
            adj[i].append(h - 1)
            adj[h - 1].append(i)
    dist = np.full((n, n), -1, dtype=np.int32)
    for s in range(n):
        dist[s, s] = 0
        q = collections.deque([s])
        while q:
            u = q.popleft()
            for v in adj[u]:
                if dist[s, v] < 0:
                    dist[s, v] = dist[s, u] + 1
                    q.append(v)
    if (dist < 0).any():
        raise ValueError("Dependency graph is disconnected")
    return dist


def is_valid_tree(heads: list[int]) -> tuple[bool, str]:
    n = len(heads)
    if n == 0:
        return False, "empty"
    if any(h < 0 or h > n for h in heads):
        return False, "head out of range"
    if sum(h == 0 for h in heads) != 1:
        return False, "root count != 1"
    for i in range(n):  # cycle check: walk to root
        seen, cur = set(), i
        while heads[cur] != 0:
            if cur in seen:
                return False, "cycle"
            seen.add(cur)
            cur = heads[cur] - 1
    return True, ""


def is_projective(heads: list[int]) -> bool:
    arcs = [(min(i + 1, h), max(i + 1, h)) for i, h in enumerate(heads) if h > 0]
    for a1, b1 in arcs:
        for a2, b2 in arcs:
            if a1 < a2 < b1 < b2:
                return False
    return True


# --------------------------------------------------------------------------------------------
# Loading
# --------------------------------------------------------------------------------------------
def parse_conllu(path: Path) -> list[Sentence]:
    sents, toks, heads, rels, sid = [], [], [], [], None
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.rstrip("\n")
            if line.startswith("# sent_id"):
                sid = line.split("=", 1)[1].strip()
            elif line.startswith("#"):
                continue
            elif not line.strip():
                if toks:
                    sents.append(Sentence(sid or f"s{len(sents)}", toks, heads, rels))
                toks, heads, rels, sid = [], [], [], None
            else:
                cols = line.split("\t")
                if len(cols) != 10:
                    raise ValueError(f"Malformed CoNLL-U line in {path}: {line[:80]!r}")
                if "-" in cols[0] or "." in cols[0]:  # multiword-token ranges / empty nodes
                    continue
                toks.append(cols[1])
                heads.append(int(cols[6]))
                rels.append(cols[7])
    if toks:
        sents.append(Sentence(sid or f"s{len(sents)}", toks, heads, rels))
    return sents


def _from_hf_rows(rows, split: str) -> list[Sentence]:
    out = []
    for k, r in enumerate(rows):
        keep = [i for i, h in enumerate(r["head"]) if str(h).isdigit()]  # drop MWT/empty rows if any
        out.append(Sentence(
            str(r.get("idx") or f"{split}-{k}"),
            [r["tokens"][i] for i in keep],
            [int(r["head"][i]) for i in keep],
            [str(r["deprel"][i]) for i in keep],
        ))
    return out


def _load_hf(cfg: dict) -> dict[str, list[Sentence]] | None:
    try:
        from datasets import load_dataset
    except ImportError:
        return None
    name, conf = cfg["data"]["hf_name"], cfg["data"]["hf_config"]
    attempts = [
        ("hf-script", dict(trust_remote_code=True)),
        ("hf-parquet", dict(revision="refs/convert/parquet")),
    ]
    for tag, kw in attempts:
        try:
            ds = load_dataset(name, conf, **kw)
            out = {s: _from_hf_rows(ds[HF_SPLIT[s]], s) for s in SPLITS if HF_SPLIT[s] in ds}
            if "train" in out and "test" in out:
                LOG.info("Loaded LinES from HuggingFace (%s)", tag)
                out["_source"] = tag  # type: ignore[assignment]
                return out
        except Exception as e:  # noqa: BLE001 - any failure -> next source
            LOG.warning("HF load (%s) failed: %s", tag, str(e).splitlines()[0][:160])
    return None


def _load_github(cfg: dict, data_dir: Path) -> dict[str, list[Sentence]]:
    raw = data_dir / "raw"
    raw.mkdir(parents=True, exist_ok=True)
    out = {}
    for s in SPLITS:
        fn = raw / f"en_lines-ud-{s}.conllu"
        if not fn.is_file() or fn.stat().st_size == 0:
            url = cfg["data"]["github_url"].format(split=s)
            LOG.info("Downloading %s", url)
            try:
                urllib.request.urlretrieve(url, fn)
            except Exception as e:
                fn.unlink(missing_ok=True)
                raise RuntimeError(
                    f"Could not download {url} ({e}). Place the CoNLL-U file manually at {fn}."
                ) from e
        out[s] = parse_conllu(fn)
    LOG.info("Loaded LinES from UD GitHub CoNLL-U files (%s)", raw)
    out["_source"] = "ud-github-" + cfg["data"]["github_url"].split("/")[-2]  # type: ignore[assignment]
    return out


def load_lines(cfg: dict) -> dict[str, list[Sentence]]:
    """Return {'train','dev','test'} -> list[Sentence], validated. Cached in-process."""
    key = id(cfg)
    if key in _CACHE:
        return _CACHE[key]
    data_dir = Path(cfg["paths"]["data_dir"])
    src = cfg["data"].get("source", "auto")
    data = None
    if src in ("auto", "hf"):
        data = _load_hf(cfg)
        if data is None and src == "hf":
            raise RuntimeError("HuggingFace loading requested but failed")
    if data is None:
        data = _load_github(cfg, data_dir)
    source = data.pop("_source")
    for s in list(data):
        valid, bad = [], collections.Counter()
        for sent in data[s]:
            ok, why = is_valid_tree(sent.heads)
            if ok and len(sent.tokens) == len(sent.heads) == len(sent.deprels):
                valid.append(sent)
            else:
                bad[why or "length mismatch"] += 1
        if bad:
            LOG.warning("%s: dropped %d invalid sentences %s", s, sum(bad.values()), dict(bad))
        data[s] = valid
        LOG.info("%-5s : %5d sentences, %6d tokens", s, len(valid), sum(map(len, valid)))
    data["_source"] = source  # type: ignore[assignment]
    _CACHE[key] = data
    return data


_CACHE: dict[int, dict] = {}


# --------------------------------------------------------------------------------------------
# Statistics
# --------------------------------------------------------------------------------------------
def dataset_report(data: dict, out_dir: Path) -> dict:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    out_dir.mkdir(parents=True, exist_ok=True)
    stats: dict = {"source": data["_source"], "splits": {}}
    train_set = {tuple(s.tokens) for s in data["train"]}
    for s in SPLITS:
        sents = data[s]
        lens = np.array([len(x) for x in sents])
        toks = [t for x in sents for t in x.tokens]
        seqs = [tuple(x.tokens) for x in sents]
        stats["splits"][s] = {
            "sentences": len(sents),
            "tokens": int(lens.sum()),
            "types_lowercased": len({t.lower() for t in toks}),
            "len_mean": float(lens.mean()), "len_median": float(np.median(lens)),
            "len_min": int(lens.min()), "len_max": int(lens.max()),
            "duplicate_sentences_within_split": len(seqs) - len(set(seqs)),
            "sentences_also_in_train": (sum(q in train_set for q in seqs) if s != "train" else None),
            "non_projective": int(sum(not is_projective(x.heads) for x in sents)),
            "punct_token_ratio": float(np.mean([p for x in sents for p in x.punct_mask])),
            "max_tree_depth_mean": float(np.mean([x.depths.max() for x in sents])),
        }
    rel = collections.Counter(d.split(":")[0] for x in data["train"] for d in x.deprels)
    stats["train_deprel_top15"] = rel.most_common(15)
    save_json(stats, out_dir / "dataset_stats.json")

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    for s in SPLITS:
        ax[0].hist([len(x) for x in data[s]], bins=40, alpha=0.5, label=s, density=True)
    ax[0].set(title="Sentence length (tokens)", xlabel="tokens", ylabel="density")
    ax[0].legend()
    names, counts = zip(*rel.most_common(15))
    ax[1].barh(names[::-1], counts[::-1], color="#4C72B0")
    ax[1].set(title="Train: top-15 dependency relations", xlabel="count")
    for s in SPLITS:
        ax[2].hist([x.depths.max() for x in data[s]], bins=range(0, 20), alpha=0.5, label=s, density=True)
    ax[2].set(title="Max tree depth per sentence", xlabel="depth")
    ax[2].legend()
    fig.tight_layout()
    fig.savefig(out_dir / "dataset_stats.png", dpi=130)
    plt.close(fig)
    LOG.info("Dataset stats -> %s", out_dir / "dataset_stats.json")
    for s in SPLITS:
        d = stats["splits"][s]
        LOG.info("  %-5s %5d sents | len %.1f±(max %d) | dup %d | in-train %s | non-proj %d",
                 s, d["sentences"], d["len_mean"], d["len_max"],
                 d["duplicate_sentences_within_split"], d["sentences_also_in_train"], d["non_projective"])
    return stats


In [ ]:
%%writefile src/p5/benchmarks.py
"""WordSim-353 and Google analogy benchmarks (shared, identical protocol for Tasks 1 and 3)."""
from __future__ import annotations

import re
import urllib.request
from pathlib import Path

import numpy as np
from scipy.stats import spearmanr

from .utils import LOG


def _fetch(urls: list[str], dest: Path) -> Path:
    if dest.is_file() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    errs = []
    for u in urls:
        try:
            LOG.info("Downloading %s", u)
            urllib.request.urlretrieve(u, dest)
            if dest.stat().st_size > 0:
                return dest
        except Exception as e:  # noqa: BLE001
            errs.append(f"{u}: {e}")
            dest.unlink(missing_ok=True)
    raise RuntimeError(f"Could not obtain {dest.name}. Tried:\n  " + "\n  ".join(errs)
                       + f"\nPlace the file manually at {dest}.")


def load_wordsim(cfg: dict) -> list[tuple[str, str, float]]:
    path = Path(cfg["benchmarks"].get("wordsim_path") or "")
    if not path.is_file():
        path = _fetch(cfg["benchmarks"]["wordsim_urls"], Path(cfg["paths"]["data_dir"]) / "wordsim353.tsv")
    pairs = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.startswith("#") or not line.strip():
                continue
            parts = line.split("\t") if "\t" in line else re.split(r"[,\s]+", line.strip())
            if len(parts) < 3:
                continue
            try:
                pairs.append((parts[0].strip().lower(), parts[1].strip().lower(), float(parts[2])))
            except ValueError:  # header line such as "Word 1  Word 2  Human (mean)"
                continue
    if len(pairs) != 353:
        LOG.warning("WordSim-353 file has %d pairs (expected 353)", len(pairs))
    return pairs


def load_analogy(cfg: dict) -> list[tuple[str, str, str, str, str]]:
    path = Path(cfg["benchmarks"].get("analogy_path") or "")
    if not path.is_file():
        path = _fetch(cfg["benchmarks"]["analogy_urls"], Path(cfg["paths"]["data_dir"]) / "questions-words.txt")
    qs, section = [], "unknown"
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            if line.startswith(":"):
                section = line[1:].strip()
                continue
            w = line.lower().split()
            if len(w) == 4:
                qs.append((section, *w))
    if len(qs) != 19544:
        LOG.warning("Analogy file has %d questions (expected 19544)", len(qs))
    return qs


def normalize(M: np.ndarray) -> np.ndarray:
    M = np.asarray(M, dtype=np.float32)
    n = np.linalg.norm(M, axis=1, keepdims=True)
    return M / np.maximum(n, 1e-12)


def eval_wordsim(vocab: dict[str, int], M: np.ndarray, pairs) -> dict:
    """Spearman between cosine similarity and human scores.

    `spearman_covered`: pairs where both words have vectors (the standard number).
    `spearman_all353`: all 353 pairs, OOV pairs get similarity 0 (strict version).
    """
    Mn = normalize(M)
    model, human, cov_mask = [], [], []
    for a, b, h in pairs:
        ok = a in vocab and b in vocab
        model.append(float(Mn[vocab[a]] @ Mn[vocab[b]]) if ok else 0.0)
        human.append(h)
        cov_mask.append(ok)
    model, human, cov_mask = np.array(model), np.array(human), np.array(cov_mask)
    rho_cov = spearmanr(model[cov_mask], human[cov_mask]).statistic if cov_mask.sum() > 2 else float("nan")
    rho_all = spearmanr(model, human).statistic
    return {"n_pairs": len(pairs), "n_covered": int(cov_mask.sum()),
            "spearman_covered": float(rho_cov), "spearman_all353": float(rho_all),
            "_model_scores": model, "_human": human, "_covered": cov_mask}


def eval_analogy(vocab: dict[str, int], M: np.ndarray, questions, batch: int = 2048) -> dict:
    """3CosAdd vector-offset method, query words excluded from candidates.

    Accuracy is reported over all 19,544 questions (OOV questions count as wrong) and over the
    covered subset, overall and split into semantic vs syntactic ('gram*') sections.
    """
    Mn = normalize(M)
    cov_idx, correct_all = [], np.zeros(len(questions), dtype=bool)
    abcd = []
    for qi, (_, a, b, c, d) in enumerate(questions):
        if all(w in vocab for w in (a, b, c, d)):
            cov_idx.append(qi)
            abcd.append((vocab[a], vocab[b], vocab[c], vocab[d]))
    if abcd:
        abcd = np.array(abcd)
        for s in range(0, len(abcd), batch):
            q = abcd[s:s + batch]
            target = Mn[q[:, 1]] - Mn[q[:, 0]] + Mn[q[:, 2]]
            scores = target @ Mn.T
            rows = np.arange(len(q))
            for k in range(3):
                scores[rows, q[:, k]] = -np.inf
            pred = scores.argmax(1)
            correct_all[np.array(cov_idx[s:s + batch])] = pred == q[:, 3]
    cov = np.zeros(len(questions), dtype=bool)
    cov[cov_idx] = True
    syn = np.array([q[0].startswith("gram") for q in questions])

    def acc(mask):
        return float(correct_all[mask].mean()) if mask.sum() else float("nan")

    return {
        "n_questions": len(questions), "n_covered": int(cov.sum()),
        "acc_all": acc(np.ones_like(cov)), "acc_covered": acc(cov),
        "acc_all_semantic": acc(~syn), "acc_all_syntactic": acc(syn),
        "acc_covered_semantic": acc(cov & ~syn), "acc_covered_syntactic": acc(cov & syn),
        "n_covered_semantic": int((cov & ~syn).sum()), "n_covered_syntactic": int((cov & syn).sum()),
        "_correct": correct_all, "_covered": cov,
    }


def public(d: dict) -> dict:
    """Strip private arrays (keys starting with '_') before saving to JSON."""
    return {k: v for k, v in d.items() if not k.startswith("_")}


In [ ]:
%%writefile src/p5/structural.py
"""MST decoding and undirected attachment scoring (UUAS), shared by Tasks 1, 2 and 3.

Evaluation follows Hewitt & Manning (2019): punctuation tokens are removed, the MST is built over
the remaining words, and gold edges are those whose both endpoints are non-punctuation.
The score is micro-averaged (total correct edges / total gold edges) over the evaluated sentences.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np

from .benchmarks import normalize
from .data import Sentence
from .utils import LOG, load_json, save_json


def prim_mst(dist: np.ndarray) -> set[tuple[int, int]]:
    """Minimum spanning tree of a dense symmetric distance matrix (Prim, O(n^2)).

    Implemented directly (not scipy.sparse.csgraph) because scipy treats 0 entries as missing
    edges, which would silently break trees when two words have identical vectors.
    """
    n = dist.shape[0]
    if n < 2:
        return set()
    in_tree = np.zeros(n, dtype=bool)
    in_tree[0] = True
    best = dist[0].astype(np.float64).copy()
    parent = np.zeros(n, dtype=int)
    edges = set()
    for _ in range(n - 1):
        cand = np.where(in_tree, np.inf, best)
        v = int(np.argmin(cand))
        edges.add((min(v, parent[v]), max(v, parent[v])))
        in_tree[v] = True
        upd = (~in_tree) & (dist[v] < best)
        best[upd] = dist[v][upd]
        parent[upd] = v
    return edges


def gold_edges(sent: Sentence, keep: np.ndarray) -> set[tuple[int, int]]:
    """Undirected gold edges between kept tokens, re-indexed to positions in the kept subset."""
    pos = -np.ones(len(sent), dtype=int)
    pos[keep] = np.arange(keep.sum())
    out = set()
    for i, h in enumerate(sent.heads):
        if h > 0 and keep[i] and keep[h - 1]:
            a, b = pos[i], pos[h - 1]
            out.add((min(a, b), max(a, b)))
    return out


def uuas_for_distance(sent: Sentence, dist: np.ndarray) -> tuple[int, int]:
    """(correct, total) undirected attachments for one sentence given a full n x n distance matrix."""
    keep = ~sent.punct_mask
    if keep.sum() < 2:
        return 0, 0
    sub = dist[np.ix_(keep, keep)]
    pred = prim_mst(sub)
    gold = gold_edges(sent, keep)
    return len(pred & gold), len(gold)


# --------------------------------------------------------------------------------------------
# 20-sentence structural check with static vectors (Task 1 bonus, Task 3 step 3)
# --------------------------------------------------------------------------------------------
def get_structural_sample(cfg: dict, test: list[Sentence]) -> list[int]:
    """Seeded sample of test-sentence indices, saved once and reused so Tasks 1 and 3 match."""
    path = Path(cfg["paths"]["output_dir"]) / "structural_sample.json"
    n, seed = cfg["structural_check"]["n_sentences"], cfg["structural_check"]["seed"]
    if path.is_file():
        saved = load_json(path)
        if saved["n"] == n and saved["seed"] == seed and saved["n_test"] == len(test):
            return saved["indices"]
    eligible = [i for i, s in enumerate(test) if (~s.punct_mask).sum() >= 2]
    rng = np.random.default_rng(seed)
    idx = sorted(int(i) for i in rng.choice(eligible, size=min(n, len(eligible)), replace=False))
    save_json({"n": n, "seed": seed, "n_test": len(test), "indices": idx,
               "sent_ids": [test[i].sid for i in idx],
               "texts": [" ".join(test[i].tokens) for i in idx]}, path)
    LOG.info("Saved structural-check sample (%d sentences) -> %s", len(idx), path)
    return idx


def static_vector_check(sents: list[Sentence], vocab: dict[str, int], M: np.ndarray) -> dict:
    """MST over pairwise cosine distances of static word vectors vs. gold trees.

    Out-of-vocabulary words get a zero vector, i.e. cosine distance 1 to every other word
    (no information); the OOV rate is reported alongside the score.
    """
    Mn = normalize(M)
    tot_c = tot_g = n_tok = n_oov = 0
    per = []
    for s in sents:
        X = np.zeros((len(s), Mn.shape[1]), dtype=np.float32)
        for i, w in enumerate(s.tokens):
            j = vocab.get(w.lower())
            if j is None:
                n_oov += 1
            else:
                X[i] = Mn[j]
        n_tok += len(s)
        dist = 1.0 - X @ X.T
        np.fill_diagonal(dist, 0.0)
        c, g = uuas_for_distance(s, dist)
        tot_c, tot_g = tot_c + c, tot_g + g
        per.append(c / g if g else float("nan"))
    return {"uuas": tot_c / max(tot_g, 1), "correct": tot_c, "gold_edges": tot_g,
            "per_sentence": per, "oov_token_rate": n_oov / max(n_tok, 1)}


def baselines(sents: list[Sentence], seed: int, n_random: int = 100) -> dict:
    """Reference points: linear chain (left-to-right adjacency) and random spanning trees."""
    tot_g = lin_c = 0
    rand_c = np.zeros(n_random)
    rng = np.random.default_rng(seed)
    per_lin = []
    for s in sents:
        keep = ~s.punct_mask
        m = int(keep.sum())
        if m < 2:
            continue
        gold = gold_edges(s, keep)
        tot_g += len(gold)
        chain = {(i, i + 1) for i in range(m - 1)}
        lin_c += len(chain & gold)
        per_lin.append(len(chain & gold) / len(gold) if gold else float("nan"))
        for r in range(n_random):  # random tree: MST over random distances
            d = rng.random((m, m))
            d = (d + d.T) / 2
            rand_c[r] += len(prim_mst(d) & gold)
    return {"linear_chain_uuas": lin_c / tot_g, "random_tree_uuas": float((rand_c / tot_g).mean()),
            "linear_chain_per_sentence": per_lin}


In [ ]:
%%writefile src/p5/task1_ppmi.py
"""Task 1 - Baseline: PPMI + SVD static vectors from LinES train (raw words only).

Pipeline: symmetric +/-5 word co-occurrence counts (within sentence, unweighted) -> PPMI ->
truncated SVD to 100 dims -> WordSim-353 / Google analogy -> MST structural check (bonus).
"""
from __future__ import annotations

import argparse
import collections
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
from sklearn.decomposition import TruncatedSVD

from .benchmarks import eval_analogy, eval_wordsim, load_analogy, load_wordsim, public
from .data import dataset_report, load_lines
from .structural import baselines, get_structural_sample, static_vector_check
from .utils import LOG, ensure_dirs, load_config, save_json, set_seed, setup_logging


def build_vocab(sents) -> dict[str, int]:
    """Lower-cased word types of the training split, frequency-sorted (ties alphabetical)."""
    cnt = collections.Counter(t.lower() for s in sents for t in s.tokens)
    words = sorted(cnt, key=lambda w: (-cnt[w], w))
    return {w: i for i, w in enumerate(words)}


def cooccurrence(sents, vocab: dict[str, int], window: int) -> sp.csr_matrix:
    """Symmetric word-word counts: every pair within `window` positions in the same sentence."""
    rows, cols = [], []
    for s in sents:
        ids = [vocab[t.lower()] for t in s.tokens]
        n = len(ids)
        for i in range(n):
            for j in range(max(0, i - window), min(n, i + window + 1)):
                if j != i:
                    rows.append(ids[i])
                    cols.append(ids[j])
    V = len(vocab)
    C = sp.coo_matrix((np.ones(len(rows), dtype=np.float64), (rows, cols)), shape=(V, V)).tocsr()
    C.sum_duplicates()
    assert (abs(C - C.T) > 0).nnz == 0, "co-occurrence matrix must be symmetric"
    return C


def ppmi(C: sp.csr_matrix, alpha: float = 1.0) -> sp.csr_matrix:
    """PPMI(w,c) = max(0, log P(w,c) / (P(w) P_alpha(c))). alpha=1 is plain PPMI;
    alpha=0.75 is context-distribution smoothing (Levy, Goldberg & Dagan 2015)."""
    C = C.tocoo()
    total = C.sum()
    row = np.asarray(C.sum(axis=1)).ravel()
    col = np.asarray(C.sum(axis=0)).ravel() ** alpha
    p_w = row / total
    p_c = col / col.sum()
    pmi = np.log((C.data / total) / (p_w[C.row] * p_c[C.col]))
    keep = pmi > 0
    return sp.csr_matrix((pmi[keep], (C.row[keep], C.col[keep])), shape=C.shape)


def svd_vectors(P: sp.csr_matrix, dim: int, power: float, seed: int) -> np.ndarray:
    """Truncated SVD; word vectors = U * S^power (power=1 -> U*Sigma, the standard choice)."""
    dim = min(dim, P.shape[0] - 1)
    svd = TruncatedSVD(n_components=dim, algorithm="randomized", n_iter=15, random_state=seed)
    US = svd.fit_transform(P)
    S = svd.singular_values_
    U = US / np.maximum(S, 1e-12)
    return (U * S ** power).astype(np.float32)


def save_vectors(vocab: dict[str, int], M: np.ndarray, out: Path) -> None:
    out.mkdir(parents=True, exist_ok=True)
    np.save(out / "vectors.npy", M)
    words = sorted(vocab, key=vocab.get)
    (out / "vocab.txt").write_text("\n".join(words) + "\n", encoding="utf-8")
    with open(out / "vectors.w2v.txt", "w", encoding="utf-8") as f:  # word2vec text format
        f.write(f"{M.shape[0]} {M.shape[1]}\n")
        for w in words:
            f.write(w + " " + " ".join(f"{x:.5f}" for x in M[vocab[w]]) + "\n")


def load_vectors(d: Path) -> tuple[dict[str, int], np.ndarray]:
    words = (d / "vocab.txt").read_text(encoding="utf-8").splitlines()
    M = np.load(d / "vectors.npy")
    if len(words) != M.shape[0]:
        raise ValueError(f"vocab/vector size mismatch in {d}")
    return {w: i for i, w in enumerate(words)}, M


def run(cfg: dict) -> dict:
    paths = ensure_dirs(cfg)
    out = paths["output_dir"] / "task1"
    out.mkdir(parents=True, exist_ok=True)
    set_seed(cfg["seed"])
    t1 = cfg["task1"]

    data = load_lines(cfg)
    dataset_report(data, paths["output_dir"] / "dataset")
    train, test = data["train"], data["test"]
    ws, an = load_wordsim(cfg), load_analogy(cfg)

    vocab = build_vocab(train)
    C = cooccurrence(train, vocab, t1["window"])
    LOG.info("Vocab %d types | co-occurrence nnz %d | total count %d", len(vocab), C.nnz, int(C.sum()))
    sample_idx = get_structural_sample(cfg, test)
    sample = [test[i] for i in sample_idx]

    # Main configuration (as specified: plain PPMI, 100-d SVD, U*Sigma) + small ablation grid
    rows, main = [], None
    for alpha in t1["ablation_alpha"]:
        P = ppmi(C, alpha)
        for power in t1["ablation_svd_power"]:
            M = svd_vectors(P, t1["dim"], power, cfg["seed"])
            w, a = eval_wordsim(vocab, M, ws), eval_analogy(vocab, M, an)
            st = static_vector_check(sample, vocab, M)
            is_main = alpha == t1["alpha"] and power == t1["svd_power"]
            rows.append({"alpha": alpha, "svd_power": power, "main": is_main,
                         "ws353_spearman_covered": w["spearman_covered"],
                         "ws353_spearman_all353": w["spearman_all353"],
                         "analogy_acc_all": a["acc_all"], "analogy_acc_covered": a["acc_covered"],
                         "struct_uuas": st["uuas"]})
            if is_main:
                main = (P, M, w, a, st)
            LOG.info("alpha=%.2f p=%.1f | WS353 rho=%.4f (cov %d) | analogy %.4f (cov %.4f) | MST UUAS %.4f%s",
                     alpha, power, w["spearman_covered"], w["n_covered"], a["acc_all"], a["acc_covered"],
                     st["uuas"], "  <- main" if is_main else "")
    if main is None:
        raise ValueError("task1.alpha / task1.svd_power must appear in the ablation lists")
    P, M, w, a, st = main
    save_vectors(vocab, M, paths["checkpoint_dir"] / "task1")
    pd.DataFrame(rows).to_csv(out / "ablation.csv", index=False)
    np.save(out / "ws353_model_scores.npy", w["_model_scores"])
    np.save(out / "analogy_correct.npy", a["_correct"])

    base = baselines(sample, cfg["seed"])
    res = {
        "config": {"window": t1["window"], "alpha": t1["alpha"], "dim": M.shape[1],
                   "svd_power": t1["svd_power"], "lowercase": True, "min_count": 1},
        "data_source": data["_source"],
        "vocab_size": len(vocab), "cooc_nnz": int(C.nnz), "ppmi_nnz": int(P.nnz),
        "train_tokens": int(sum(map(len, train))),
        "wordsim353": public(w), "analogy": public(a),
        "structural_check": {k: v for k, v in st.items()},
        "structural_baselines": base,
    }
    save_json(res, out / "results.json")
    _plot(rows, out)
    LOG.info("TASK 1 | WordSim-353 Spearman %.4f on %d/353 covered pairs (all-353: %.4f)",
             w["spearman_covered"], w["n_covered"], w["spearman_all353"])
    LOG.info("TASK 1 | Google analogy accuracy %.4f overall (%d/19544 covered; %.4f on covered)",
             a["acc_all"], a["n_covered"], a["acc_covered"])
    LOG.info("TASK 1 | MST-vs-gold UUAS on %d test sentences: %.4f (linear chain %.4f, random %.4f)",
             len(sample), st["uuas"], base["linear_chain_uuas"], base["random_tree_uuas"])
    return res


def _plot(rows, out: Path) -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    df = pd.DataFrame(rows)
    df["label"] = df.apply(lambda r: f"a={r.alpha}, p={r.svd_power}" + (" *" if r.main else ""), axis=1)
    fig, ax = plt.subplots(1, 3, figsize=(14, 3.8))
    for k, (col, t) in enumerate([("ws353_spearman_covered", "WordSim-353 Spearman (covered)"),
                                  ("analogy_acc_covered", "Analogy acc (covered)"),
                                  ("struct_uuas", "MST UUAS (20 test sents)")]):
        ax[k].bar(df["label"], df[col], color=["#C44E52" if m else "#8C8C8C" for m in df["main"]])
        ax[k].set_title(t)
        ax[k].tick_params(axis="x", rotation=30)
    fig.suptitle("Task 1 PPMI+SVD (* = specified configuration)")
    fig.tight_layout()
    fig.savefig(out / "task1_ablation.png", dpi=130)
    plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="config.yaml")
    args = ap.parse_args()
    cfg = load_config(args.config)
    setup_logging(Path(cfg["paths"]["output_dir"]) / "run.log")
    run(cfg)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/p5/features.py
"""Frozen BERT feature extraction, word-level, all layers, cached to disk.

Each word's vector at layer l is the mean of its WordPiece vectors (as in Hewitt & Manning's
code). Features are stored per split and layer as one float16 array of shape (n_words, hidden)
plus per-sentence offsets, so Task 2 (probes) and Task 3 (pooling) share one extraction pass.
"""
from __future__ import annotations

from pathlib import Path

import numpy as np
import torch

from .data import Sentence
from .utils import LOG, amp_dtype, gpu_mem_gb, load_json, save_json


def _cache_dir(cfg: dict) -> Path:
    name = Path(cfg["bert"]["model_name"]).name.replace("/", "_")
    d = Path(cfg["paths"]["cache_dir"]) / f"features_{name}"
    d.mkdir(parents=True, exist_ok=True)
    return d


def _auto_batch(cfg: dict, device) -> int:
    bs = cfg["bert"].get("extract_batch_size", "auto")
    if bs != "auto":
        return int(bs)
    mem = gpu_mem_gb(device)
    return 64 if mem >= 20 else 32 if mem >= 10 else 16 if mem > 0 else 8


def load_bert(cfg: dict, device):
    from transformers import AutoModel, AutoTokenizer

    name = cfg["bert"]["model_name"]
    tok = AutoTokenizer.from_pretrained(name, use_fast=True)
    if not tok.is_fast:
        raise RuntimeError("A fast tokenizer is required (word_ids alignment)")
    model = AutoModel.from_pretrained(name, output_hidden_states=True).to(device).eval()
    for p in model.parameters():
        p.requires_grad_(False)
    dev = next(model.parameters()).device
    assert dev.type == device.type, f"model on {dev}, expected {device}"
    LOG.info("Loaded %s on %s | layers %d | hidden %d", name, dev,
             model.config.num_hidden_layers, model.config.hidden_size)
    return tok, model


@torch.inference_mode()
def extract_split(cfg: dict, split: str, sents: list[Sentence], device) -> dict:
    """Extract (or load cached) word features for every layer of one split.

    Returns meta: {'offsets', 'lengths', 'skipped', 'n_layers', 'hidden', 'dir'}.
    """
    d = _cache_dir(cfg)
    meta_path = d / f"{split}_meta.json"
    sig = [len(s) for s in sents]
    if meta_path.is_file():
        meta = load_json(meta_path)
        files_ok = all((d / f"{split}_L{l}.npy").is_file() for l in range(meta["n_layers"] + 1))
        if meta["lengths"] == sig and files_ok:
            LOG.info("Using cached %s features (%s)", split, d)
            meta["dir"] = str(d)
            return meta

    tok, model = load_bert(cfg, device)
    n_layers, H = model.config.num_hidden_layers, model.config.hidden_size
    max_len = min(getattr(tok, "model_max_length", 512), model.config.max_position_embeddings)
    bs = _auto_batch(cfg, device)
    dt = amp_dtype(device) if cfg["bert"].get("mixed_precision", True) else None
    LOG.info("Extracting %s: %d sentences | batch %d | amp %s", split, len(sents), bs, dt)

    def clean(w: str) -> str:  # words that tokenize to nothing (e.g. control chars) -> [UNK]
        return w if tok.tokenize(w) else tok.unk_token

    order = np.argsort([-len(s) for s in sents], kind="stable")  # length-sorted -> less padding
    total_words = sum(sig)
    offsets = np.concatenate([[0], np.cumsum(sig)]).astype(np.int64)
    feats = [np.zeros((total_words, H), dtype=np.float16) for _ in range(n_layers + 1)]
    skipped = []
    for b in range(0, len(order), bs):
        idx = [int(i) for i in order[b:b + bs]]
        words = [[clean(w) for w in sents[i].tokens] for i in idx]
        enc = tok(words, is_split_into_words=True, return_tensors="pt", padding=True, truncation=False)
        keep_rows = [r for r in range(len(idx)) if int(enc["attention_mask"][r].sum()) <= max_len]
        for r in range(len(idx)):
            if r not in keep_rows:
                skipped.append(idx[r])
                LOG.warning("Sentence %s exceeds %d subwords - skipped", sents[idx[r]].sid, max_len)
        if not keep_rows:
            continue
        L = max(int(enc["attention_mask"][r].sum()) for r in keep_rows)
        inputs = {k: v[keep_rows, :L].to(device) for k, v in enc.items()}
        with torch.autocast(device.type, dtype=dt, enabled=dt is not None):
            hs = model(**inputs).hidden_states  # tuple(n_layers+1) of (b, L, H)
        hs = torch.stack(hs, 0).float()  # (layers, b, L, H)
        for bi, r in enumerate(keep_rows):
            wid = enc.word_ids(r)[:L]
            n_w = len(words[r])
            map_ = torch.zeros(n_w, L, device=device)
            for t, w in enumerate(wid):
                if w is not None:
                    map_[w, t] = 1.0
            cnt = map_.sum(1, keepdim=True)
            if (cnt == 0).any():
                raise RuntimeError(f"word without subwords in {sents[idx[r]].sid}")
            pooled = torch.einsum("wt,ltd->lwd", map_ / cnt, hs[:, bi])  # mean of subwords
            s0, s1 = offsets[idx[r]], offsets[idx[r] + 1]
            arr = pooled.to(torch.float16).cpu().numpy()
            for l in range(n_layers + 1):
                feats[l][s0:s1] = arr[l]
    for l in range(n_layers + 1):
        np.save(d / f"{split}_L{l}.npy", feats[l])
    meta = {"offsets": offsets.tolist(), "lengths": sig, "skipped": sorted(skipped),
            "n_layers": n_layers, "hidden": H, "model": cfg["bert"]["model_name"]}
    save_json(meta, meta_path)
    meta["dir"] = str(d)
    del model
    if device.type == "cuda":
        torch.cuda.empty_cache()
    LOG.info("Saved %s features: %d words x %d dims x %d layers (%d skipped sents)",
             split, total_words, H, n_layers + 1, len(skipped))
    return meta


def layer_features(meta: dict, split: str, layer: int, mmap: bool = True) -> np.ndarray:
    return np.load(Path(meta["dir"]) / f"{split}_L{layer}.npy", mmap_mode="r" if mmap else None)


In [ ]:
%%writefile src/p5/task2_probe.py
"""Task 2 - Structural probe on frozen bert-base-uncased (Hewitt & Manning, NAACL 2019).

For each layer l = 1..12 and rank k in {32, 64, 128}, train
  * a distance probe  B: ||B(h_i - h_j)||^2  ~ tree distance d(i, j)
  * a depth probe     B: ||B h_i||^2        ~ tree depth   |i|
with L1 loss normalised by sentence length (squared for distance) as in the paper's code.
Adam, lr 1e-3, batch 20 sentences, 30 epochs, on LinES train. The final-epoch probe is kept.
Evaluation: UUAS of the MST over predicted distances, depth Spearman (DSpr, averaged over
sentence lengths 5-50 then macro-averaged across lengths), plus root accuracy and distance DSpr.
The best layer l* is selected on the dev split (no test leakage) and reported on test.
"""
from __future__ import annotations

import argparse
import collections
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import rankdata, spearmanr

from .data import Sentence, load_lines
from .features import extract_split, layer_features
from .structural import uuas_for_distance
from .utils import LOG, ensure_dirs, get_device, load_config, save_json, set_seed, setup_logging


class LinearProbe(nn.Module):
    def __init__(self, hidden: int, rank: int, kind: str):
        super().__init__()
        assert kind in ("distance", "depth")
        self.kind = kind
        self.proj = nn.Parameter(torch.empty(hidden, rank))
        nn.init.uniform_(self.proj, -0.05, 0.05)  # same init as the reference implementation

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        t = x @ self.proj  # (b, L, k)
        if self.kind == "depth":
            return (t * t).sum(-1)  # (b, L)
        diff = t.unsqueeze(2) - t.unsqueeze(1)  # (b, L, L, k)
        return (diff * diff).sum(-1)  # (b, L, L)


class SplitTensors:
    """GPU-resident word features (one layer) + padded gold targets for one split."""

    def __init__(self, sents: list[Sentence], meta: dict, split: str, device):
        skip = set(meta["skipped"])
        self.idx = [i for i in range(len(sents)) if i not in skip]
        self.sents = [sents[i] for i in self.idx]
        self.lens = torch.tensor([len(s) for s in self.sents], device=device)
        self.Lmax = int(self.lens.max())
        off = meta["offsets"]
        n = len(self.sents)
        self.gather = torch.full((n, self.Lmax), -1, dtype=torch.long)
        self.dist = torch.zeros((n, self.Lmax, self.Lmax), dtype=torch.float16)
        self.depth = torch.zeros((n, self.Lmax), dtype=torch.float16)
        for k, i in enumerate(self.idx):
            m = len(sents[i])
            self.gather[k, :m] = torch.arange(off[i], off[i] + m)
            self.dist[k, :m, :m] = torch.from_numpy(sents[i].distances.astype(np.float16))
            self.depth[k, :m] = torch.from_numpy(sents[i].depths.astype(np.float16))
        self.gather, self.dist, self.depth = self.gather.to(device), self.dist.to(device), self.depth.to(device)
        self.meta, self.split, self.device, self.X = meta, split, device, None

    def load_layer(self, layer: int):
        arr = np.asarray(layer_features(self.meta, self.split, layer, mmap=False))
        zero = np.zeros((1, arr.shape[1]), dtype=arr.dtype)  # padding row at index -1 -> last
        self.X = torch.from_numpy(np.concatenate([arr, zero])).to(self.device)

    def batch(self, rows: torch.Tensor):
        L = int(self.lens[rows].max())
        g = self.gather[rows, :L]
        x = self.X[g].float()  # index -1 -> zero padding row
        mask = g >= 0
        return x, mask, self.dist[rows, :L, :L].float(), self.depth[rows, :L].float(), self.lens[rows].float()


def probe_loss(pred, gold, mask, lens, kind):
    if kind == "distance":
        m = (mask.unsqueeze(1) & mask.unsqueeze(2)).float()
        per = ((pred - gold).abs() * m).sum((1, 2)) / (lens ** 2)
    else:
        per = ((pred - gold).abs() * mask.float()).sum(1) / lens
    return per.mean()


def train_probe(tr: SplitTensors, kind: str, rank: int, cfg: dict, device, gen: torch.Generator):
    p2 = cfg["task2"]
    probe = LinearProbe(tr.X.shape[1], rank, kind).to(device)
    opt = torch.optim.Adam(probe.parameters(), lr=p2["lr"])
    n, bs = len(tr.sents), p2["batch_size"]
    history = []
    for ep in range(p2["epochs"]):
        perm = torch.randperm(n, generator=gen).to(device)
        tot = 0.0
        for b in range(0, n, bs):
            rows = perm[b:b + bs]
            x, mask, dist, depth, lens = tr.batch(rows)
            loss = probe_loss(probe(x), dist if kind == "distance" else depth, mask, lens, kind)
            opt.zero_grad(set_to_none=True)
            loss.backward()
            opt.step()
            tot += loss.item() * len(rows)
        history.append(tot / n)
    return probe, history


def _rowwise_spearman(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """Spearman correlation of each row of A with the same row of B (average ranks for ties)."""
    ra, rb = rankdata(A, axis=1), rankdata(B, axis=1)
    ra -= ra.mean(1, keepdims=True)
    rb -= rb.mean(1, keepdims=True)
    den = np.sqrt((ra * ra).sum(1) * (rb * rb).sum(1))
    with np.errstate(invalid="ignore", divide="ignore"):
        return np.where(den > 0, (ra * rb).sum(1) / den, np.nan)


@torch.no_grad()
def evaluate(probe: LinearProbe, ds: SplitTensors, bs: int = 64) -> dict:
    probe.eval()
    out: dict = {}
    by_len = collections.defaultdict(list)
    c_tot = g_tot = root_ok = root_n = 0
    for b in range(0, len(ds.sents), bs):
        rows = torch.arange(b, min(b + bs, len(ds.sents)), device=ds.device)
        x, mask, *_ = ds.batch(rows)
        pred = probe(x).cpu().numpy()
        for r, k in enumerate(rows.tolist()):
            s = ds.sents[k]
            n, keep = len(s), ~s.punct_mask
            if probe.kind == "distance":
                P = pred[r, :n, :n]
                c, g = uuas_for_distance(s, P)
                c_tot, g_tot = c_tot + c, g_tot + g
                G = s.distances[np.ix_(keep, keep)]
                Pk = P[np.ix_(keep, keep)]
                if 5 <= keep.sum() <= 50:
                    by_len[int(keep.sum())].append(np.nanmean(_rowwise_spearman(Pk, G)))
            else:
                p, gd = pred[r, :n], s.depths
                if keep.sum() >= 1:
                    root = s.heads.index(0)
                    if keep[root]:
                        root_n += 1
                        root_ok += int(np.argmin(np.where(keep, p, np.inf)) == root)
                if 5 <= keep.sum() <= 50:
                    by_len[int(keep.sum())].append(spearmanr(p[keep], gd[keep]).statistic)
    dspr = float(np.nanmean([np.nanmean(v) for v in by_len.values()])) if by_len else float("nan")
    if probe.kind == "distance":
        out = {"uuas": c_tot / max(g_tot, 1), "dist_dspr": dspr}
    else:
        out = {"depth_spearman": dspr, "root_acc": root_ok / max(root_n, 1)}
    probe.train()
    return out


def run(cfg: dict) -> dict:
    paths = ensure_dirs(cfg)
    out = paths["output_dir"] / "task2"
    ck = paths["checkpoint_dir"] / "task2"
    out.mkdir(parents=True, exist_ok=True)
    ck.mkdir(parents=True, exist_ok=True)
    set_seed(cfg["seed"])
    device = get_device(cfg.get("device", "auto"))
    p2 = cfg["task2"]
    data = load_lines(cfg)
    metas = {s: extract_split(cfg, s, data[s], device) for s in ("train", "dev", "test")}
    tr, dv, te = (SplitTensors(data[s], metas[s], s, device) for s in ("train", "dev", "test"))
    LOG.info("Probe data: train %d / dev %d / test %d sentences on %s",
             len(tr.sents), len(dv.sents), len(te.sents), tr.dist.device)

    rows, curves = [], {}
    layers = p2["layers"]
    for layer in layers:
        for ds in (tr, dv, te):
            ds.load_layer(layer)
        for kind in ("distance", "depth"):
            for rank in p2["ranks"]:
                f = ck / f"{kind}_L{layer:02d}_r{rank}.pt"
                t0 = time.time()
                if f.is_file() and p2.get("resume", True):
                    state = torch.load(f, map_location=device, weights_only=False)
                    probe = LinearProbe(tr.X.shape[1], rank, kind).to(device)
                    probe.load_state_dict(state["state_dict"])
                    hist = state["history"]
                else:
                    gen = torch.Generator().manual_seed(cfg["seed"] + 1000 * layer + rank)
                    torch.manual_seed(cfg["seed"] + 1000 * layer + rank)
                    probe, hist = train_probe(tr, kind, rank, cfg, device, gen)
                    torch.save({"state_dict": probe.state_dict(), "history": hist, "layer": layer,
                                "rank": rank, "kind": kind, "model": cfg["bert"]["model_name"]}, f)
                dev_m, test_m = evaluate(probe, dv), evaluate(probe, te)
                curves[f"{kind}_L{layer}_r{rank}"] = hist
                row = {"layer": layer, "kind": kind, "rank": rank, "final_train_loss": hist[-1]}
                row.update({f"dev_{k}": v for k, v in dev_m.items()})
                row.update({f"test_{k}": v for k, v in test_m.items()})
                rows.append(row)
                LOG.info("L%02d %-8s k=%-3d | loss %.4f | dev %s | test %s | %.1fs", layer, kind, rank,
                         hist[-1], _fmt(dev_m), _fmt(test_m), time.time() - t0)
    df = pd.DataFrame(rows)
    df.to_csv(out / "probe_results.csv", index=False)
    save_json(curves, out / "loss_curves.json")

    # ---- select l*: best dev UUAS (max over ranks); tie-break by dev depth Spearman ----
    dist = df[df.kind == "distance"].groupby("layer")
    dep = df[df.kind == "depth"].groupby("layer")
    summ = pd.DataFrame({
        "dev_uuas": dist["dev_uuas"].max(), "test_uuas": dist["test_uuas"].max(),
        "test_uuas_best_dev_rank": dist.apply(lambda g: g.loc[g.dev_uuas.idxmax(), "test_uuas"],
                                              include_groups=False),
        "dev_depth_spearman": dep["dev_depth_spearman"].max(),
        "test_depth_spearman": dep["test_depth_spearman"].max(),
        "test_root_acc": dep["test_root_acc"].max(),
    })
    sel = p2.get("select_on", "dev")
    key = f"{sel}_uuas"
    best = summ.sort_values([key, f"{sel}_depth_spearman"], ascending=False).index[0]
    summ.to_csv(out / "layer_summary.csv")
    res = {
        "best_layer": int(best), "selected_on": sel,
        "best_layer_by_test_uuas": int(summ["test_uuas"].idxmax()),
        "best_layer_by_test_depth_spearman": int(summ["test_depth_spearman"].idxmax()),
        "best_layer_metrics": summ.loc[best].to_dict(),
        "layer_summary": summ.reset_index().to_dict(orient="records"),
        "n_skipped_long_sentences": {s: len(metas[s]["skipped"]) for s in metas},
        "device": str(device),
    }
    save_json(res, out / "results.json")
    save_json({"best_layer": int(best)}, out / "best_layer.json")
    _plot(df, summ, int(best), out)
    LOG.info("TASK 2 | l* = %d (selected on %s) | test UUAS %.4f (dev-selected rank) | test depth Spearman %.4f",
             best, sel, summ.loc[best, "test_uuas_best_dev_rank"], summ.loc[best, "test_depth_spearman"])
    if res["best_layer_by_test_uuas"] != best:
        LOG.info("TASK 2 | note: test-argmax UUAS layer is %d", res["best_layer_by_test_uuas"])
    return res


def _fmt(m: dict) -> str:
    return " ".join(f"{k}={v:.4f}" for k, v in m.items())


def _plot(df: pd.DataFrame, summ: pd.DataFrame, best: int, out: Path) -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
    for (metric, kind, title), a in zip(
            [("uuas", "distance", "UUAS (distance probe)"),
             ("depth_spearman", "depth", "Depth-probe Spearman (DSpr, lengths 5-50)")], ax):
        sub = df[df.kind == kind]
        for ci, (rank, g) in enumerate(sub.groupby("rank")):
            g, col = g.sort_values("layer"), f"C{ci}"
            a.plot(g.layer, g[f"test_{metric}"], "o-", color=col, label=f"test, k={rank}")
            a.plot(g.layer, g[f"dev_{metric}"], ":", color=col, alpha=0.6, label=f"dev, k={rank}")
        a.axvline(best, color="k", ls="--", lw=1)
        a.text(best, a.get_ylim()[1], f" l*={best}", va="top")
        a.set(title=title, xlabel="BERT layer", xticks=sorted(df.layer.unique()))
        a.grid(alpha=0.3)
    ax[0].legend(fontsize=7, ncol=2)
    fig.suptitle("Task 2: structural probe on bert-base-uncased, UD English LinES")
    fig.tight_layout()
    fig.savefig(out / "task2_uuas_spearman_vs_layer.png", dpi=130)
    plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="config.yaml")
    args = ap.parse_args()
    cfg = load_config(args.config)
    setup_logging(Path(cfg["paths"]["output_dir"]) / "run.log")
    run(cfg)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/p5/task3_bert_static.py
"""Task 3 - BERT-derived static vectors from layer l* (Bommasani et al. 2020-style pooling).

For every occurrence of a word in LinES train, take its layer-l* contextual vector (mean of
subwords, from the Task 2 cache) and average per lower-cased word type. The vocabulary is
therefore identical to Task 1's, so both models are scored on exactly the same benchmark pairs,
analogy questions and 20 test sentences. Also reported: a 100-d PCA version (dimension-matched
to Task 1) and a sweep over all layers, to show whether semantic and syntactic answers agree.
"""
from __future__ import annotations

import argparse
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import binomtest, wilcoxon
from sklearn.decomposition import PCA

from .benchmarks import eval_analogy, eval_wordsim, load_analogy, load_wordsim, public
from .data import load_lines
from .features import extract_split, layer_features
from .structural import get_structural_sample, static_vector_check
from .task1_ppmi import build_vocab, load_vectors
from .utils import LOG, ensure_dirs, get_device, load_config, load_json, save_json, set_seed, setup_logging


def pool_static(sents, meta: dict, layer: int, vocab: dict[str, int]) -> tuple[np.ndarray, np.ndarray]:
    X = layer_features(meta, "train", layer, mmap=True)
    S = np.zeros((len(vocab), X.shape[1]), dtype=np.float64)
    cnt = np.zeros(len(vocab), dtype=np.int64)
    off, skip = meta["offsets"], set(meta["skipped"])
    for i, s in enumerate(sents):
        if i in skip:
            continue
        ids = np.fromiter((vocab[t.lower()] for t in s.tokens), dtype=np.int64, count=len(s))
        np.add.at(S, ids, np.asarray(X[off[i]:off[i + 1]], dtype=np.float64))
        np.add.at(cnt, ids, 1)
    M = (S / np.maximum(cnt, 1)[:, None]).astype(np.float32)
    return M, cnt


def bootstrap_ws_diff(scores_a, scores_b, human, covered, n_boot: int, seed: int) -> dict:
    """Paired bootstrap over covered WordSim pairs for rho(B) - rho(A)."""
    from scipy.stats import spearmanr

    idx = np.flatnonzero(covered)
    rng = np.random.default_rng(seed)
    diffs = []
    for _ in range(n_boot):
        s = rng.choice(idx, size=len(idx), replace=True)
        diffs.append(spearmanr(scores_b[s], human[s]).statistic - spearmanr(scores_a[s], human[s]).statistic)
    diffs = np.array(diffs)
    return {"delta": float(spearmanr(scores_b[idx], human[idx]).statistic
                           - spearmanr(scores_a[idx], human[idx]).statistic),
            "ci95": [float(np.nanpercentile(diffs, 2.5)), float(np.nanpercentile(diffs, 97.5))],
            "p_boot_two_sided": float(min(1.0, 2 * min((diffs <= 0).mean(), (diffs >= 0).mean())))}


def mcnemar(correct_a, correct_b, mask) -> dict:
    a, b = correct_a[mask], correct_b[mask]
    only_a, only_b = int((a & ~b).sum()), int((~a & b).sum())
    p = binomtest(only_b, only_a + only_b, 0.5).pvalue if only_a + only_b else 1.0
    return {"only_task1_correct": only_a, "only_task3_correct": only_b, "p_exact_mcnemar": float(p)}


def run(cfg: dict) -> dict:
    paths = ensure_dirs(cfg)
    out = paths["output_dir"] / "task3"
    out.mkdir(parents=True, exist_ok=True)
    set_seed(cfg["seed"])
    device = get_device(cfg.get("device", "auto"))
    t3 = cfg["task3"]
    data = load_lines(cfg)
    train, test = data["train"], data["test"]
    ws, an = load_wordsim(cfg), load_analogy(cfg)

    best = t3.get("layer") or load_json(paths["output_dir"] / "task2" / "best_layer.json")["best_layer"]
    LOG.info("Using BERT layer l* = %d", best)
    meta = extract_split(cfg, "train", train, device)  # cached by Task 2
    vocab = build_vocab(train)
    t1_dir = paths["checkpoint_dir"] / "task1"
    v1, M1 = load_vectors(t1_dir)
    if v1 != vocab:
        raise RuntimeError("Task 1 vocabulary differs from Task 3's - rerun Task 1 with the same data")

    sample = [test[i] for i in get_structural_sample(cfg, test)]
    t1_res = load_json(paths["output_dir"] / "task1" / "results.json")

    # ---- main: layer l*, raw 768-d pooled vectors ----
    M3, cnt = pool_static(train, meta, best, vocab)
    if (cnt == 0).any():
        LOG.warning("%d word types have no occurrence (skipped sentences)", int((cnt == 0).sum()))
    np.save(paths["checkpoint_dir"] / f"task3_vectors_L{best:02d}.npy", M3)
    (paths["checkpoint_dir"] / "task3_vocab.txt").write_text(
        "\n".join(sorted(vocab, key=vocab.get)) + "\n", encoding="utf-8")
    w3, a3, s3 = eval_wordsim(vocab, M3, ws), eval_analogy(vocab, M3, an), static_vector_check(sample, vocab, M3)
    M3p = PCA(n_components=min(100, M3.shape[1]), random_state=cfg["seed"]).fit_transform(M3)
    w3p, a3p, s3p = eval_wordsim(vocab, M3p, ws), eval_analogy(vocab, M3p, an), static_vector_check(sample, vocab, M3p)
    w1, a1, s1 = eval_wordsim(vocab, M1, ws), eval_analogy(vocab, M1, an), static_vector_check(sample, vocab, M1)

    # ---- paired significance vs Task 1 ----
    sig = {
        "wordsim_bootstrap": bootstrap_ws_diff(w1["_model_scores"], w3["_model_scores"], w1["_human"],
                                               w1["_covered"], t3["n_bootstrap"], cfg["seed"]),
        "analogy_mcnemar": mcnemar(a1["_correct"], a3["_correct"], a1["_covered"]),
    }
    p1, p3 = np.array(s1["per_sentence"]), np.array(s3["per_sentence"])
    ok = ~(np.isnan(p1) | np.isnan(p3))
    d = p3[ok] - p1[ok]
    sig["structural_wilcoxon"] = {
        "mean_per_sentence_delta": float(d.mean()),
        "n_better": int((d > 0).sum()), "n_worse": int((d < 0).sum()), "n_tie": int((d == 0).sum()),
        "p_wilcoxon": float(wilcoxon(d).pvalue) if (d != 0).any() else 1.0,
    }

    # ---- sweep across all layers (does the syntax-best layer also give the best semantics?) ----
    sweep = []
    for layer in range(0 if t3.get("sweep_include_layer0", True) else 1, meta["n_layers"] + 1):
        M, _ = pool_static(train, meta, layer, vocab)
        w, a, s = eval_wordsim(vocab, M, ws), eval_analogy(vocab, M, an), static_vector_check(sample, vocab, M)
        sweep.append({"layer": layer, "ws353_spearman": w["spearman_covered"], "analogy_acc_all": a["acc_all"],
                      "analogy_acc_covered": a["acc_covered"],
                      "analogy_sem_covered": a["acc_covered_semantic"],
                      "analogy_syn_covered": a["acc_covered_syntactic"], "struct_uuas": s["uuas"]})
        LOG.info("sweep L%02d | WS353 %.4f | analogy %.4f | MST UUAS %.4f", layer,
                 w["spearman_covered"], a["acc_all"], s["uuas"])
    sweep_df = pd.DataFrame(sweep)
    sweep_df.to_csv(out / "layer_sweep.csv", index=False)

    comp = pd.DataFrame([
        _row("Task 1: PPMI+SVD (100d)", w1, a1, s1),
        _row(f"Task 3: BERT L{best} mean-pooled (768d)", w3, a3, s3),
        _row(f"Task 3: BERT L{best} + PCA (100d)", w3p, a3p, s3p),
    ])
    comp.to_csv(out / "comparison.csv", index=False)
    verdict = _verdict(comp, sig, sweep_df, best)
    res = {
        "best_layer": best, "vocab_size": len(vocab),
        "wordsim353": public(w3), "analogy": public(a3), "structural_check": s3,
        "pca100": {"wordsim353": public(w3p), "analogy": public(a3p), "structural_check": s3p},
        "task1_reference": {"wordsim353": public(w1), "analogy": public(a1), "structural_check": s1},
        "task1_results_file_uuas": t1_res["structural_check"]["uuas"],
        "significance": sig, "comparison": comp.to_dict(orient="records"),
        "layer_sweep": sweep, "verdict": verdict,
    }
    save_json(res, out / "results.json")
    _plot(comp, sweep_df, best, out)
    LOG.info("\n%s", comp.to_string(index=False))
    for v in verdict["statements"]:
        LOG.info("TASK 3 | %s", v)
    return res


def _row(name, w, a, s):
    return {"model": name, "ws353_spearman_covered": w["spearman_covered"], "ws353_n_covered": w["n_covered"],
            "ws353_spearman_all353": w["spearman_all353"], "analogy_acc_all": a["acc_all"],
            "analogy_acc_covered": a["acc_covered"], "analogy_n_covered": a["n_covered"],
            "analogy_sem_covered": a["acc_covered_semantic"], "analogy_syn_covered": a["acc_covered_syntactic"],
            "struct_uuas_20": s["uuas"]}


def _verdict(comp: pd.DataFrame, sig: dict, sweep: pd.DataFrame, best: int) -> dict:
    """Data-driven statements (no hand-written numbers) for the report's discussion."""
    t1, t3 = comp.iloc[0], comp.iloc[1]
    st = []

    def cmp(metric, label, p):
        d = t3[metric] - t1[metric]
        word = "higher" if d > 0 else "lower" if d < 0 else "equal"
        sigw = "statistically significant" if p < 0.05 else "not statistically significant"
        st.append(f"{label}: BERT-L{best} {t3[metric]:.4f} vs PPMI {t1[metric]:.4f} "
                  f"({word}, delta {d:+.4f}; {sigw}, p={p:.3g}).")
        return d

    dw = cmp("ws353_spearman_covered", "WordSim-353 Spearman", sig["wordsim_bootstrap"]["p_boot_two_sided"])
    da = cmp("analogy_acc_covered", "Analogy accuracy (covered)", sig["analogy_mcnemar"]["p_exact_mcnemar"])
    ds = cmp("struct_uuas_20", "Structural MST UUAS (20 sents)", sig["structural_wilcoxon"]["p_wilcoxon"])
    sem_better = (dw > 0) + (da > 0)
    if sem_better == 2 and ds > 0:
        agree = "agree: BERT-derived vectors are better on both semantic benchmarks and the structural check"
    elif sem_better == 0 and ds <= 0:
        agree = "agree: the classical vectors are at least as good on both semantic and structural evaluations"
    else:
        agree = "do not fully agree: the semantic benchmarks and the structural check rank the two models differently"
    st.append(f"The two kinds of evaluation {agree}.")
    bw = int(sweep.loc[sweep.ws353_spearman.idxmax(), "layer"])
    bs = int(sweep.loc[sweep.struct_uuas.idxmax(), "layer"])
    st.append(f"Across pooled layers, WordSim-353 peaks at layer {bw} and the static-vector MST check peaks at "
              f"layer {bs}; the probe-selected layer is {best}.")
    return {"statements": st, "best_ws_layer_pooled": bw, "best_struct_layer_pooled": bs}


def _plot(comp: pd.DataFrame, sweep: pd.DataFrame, best: int, out: Path) -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    fig, ax = plt.subplots(1, 3, figsize=(15, 4))
    names = ["PPMI+SVD", f"BERT L{best}", f"BERT L{best}\nPCA-100"]
    cols = ["#8C8C8C", "#8172B2", "#B39DDB"]
    for a, (m, t) in zip(ax, [("ws353_spearman_covered", "WordSim-353 Spearman (covered pairs)"),
                              ("analogy_acc_covered", "Google analogy acc (covered)"),
                              ("struct_uuas_20", "MST UUAS, 20 test sentences")]):
        a.bar(names, comp[m], color=cols)
        for i, v in enumerate(comp[m]):
            a.text(i, v, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
        a.set_title(t)
    fig.suptitle("Task 3 vs Task 1 (same vocabulary, same benchmark items, same 20 sentences)")
    fig.tight_layout()
    fig.savefig(out / "task3_vs_task1.png", dpi=130)
    plt.close(fig)

    fig, a1 = plt.subplots(figsize=(8, 4.5))
    a1.plot(sweep.layer, sweep.ws353_spearman, "o-", label="WordSim-353 Spearman")
    a1.plot(sweep.layer, sweep.analogy_acc_covered, "s-", label="Analogy acc (covered)")
    a1.plot(sweep.layer, sweep.struct_uuas, "^-", label="MST UUAS (20 sents)")
    a1.axvline(best, color="k", ls="--", lw=1, label=f"l* = {best} (Task 2)")
    a1.set(xlabel="BERT layer used for pooling", ylabel="score", xticks=list(sweep.layer),
           title="Pooled static vectors: semantic vs structural quality by layer")
    a1.grid(alpha=0.3)
    a1.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(out / "task3_layer_sweep.png", dpi=130)
    plt.close(fig)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--config", default="config.yaml")
    args = ap.parse_args()
    cfg = load_config(args.config)
    setup_logging(Path(cfg["paths"]["output_dir"]) / "run.log")
    run(cfg)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile src/p5/report.py
"""Assemble outputs/REPORT.md from the JSON/CSV results of the three tasks (no hand-typed numbers)."""
from __future__ import annotations

from pathlib import Path

import pandas as pd

from .utils import LOG, load_json


def _md(df: pd.DataFrame, floatfmt: str = ".4f") -> str:
    df = df.copy()
    for c in df.columns:
        if pd.api.types.is_float_dtype(df[c]):
            df[c] = df[c].map(lambda v: format(v, floatfmt))
    cols = list(df.columns)
    lines = ["| " + " | ".join(map(str, cols)) + " |", "|" + "---|" * len(cols)]
    lines += ["| " + " | ".join(str(v) for v in r) + " |" for r in df.itertuples(index=False)]
    return "\n".join(lines)


def build(cfg: dict) -> Path:
    out = Path(cfg["paths"]["output_dir"])
    r = ["# Project 5 - From Static to Contextual: results on UD English LinES", ""]
    ds = load_json(out / "dataset" / "dataset_stats.json") if (out / "dataset" / "dataset_stats.json").is_file() else None
    if ds:
        r += [f"Data source: `{ds['source']}`", "",
              _md(pd.DataFrame([{"split": k, **{x: v[x] for x in ("sentences", "tokens", "types_lowercased",
                                 "len_mean", "len_max", "non_projective", "sentences_also_in_train")}}
                                for k, v in ds["splits"].items()])), "", "![dataset](dataset/dataset_stats.png)", ""]
    if (out / "task1" / "results.json").is_file():
        t1 = load_json(out / "task1" / "results.json")
        w, a, s, b = t1["wordsim353"], t1["analogy"], t1["structural_check"], t1["structural_baselines"]
        r += ["## Task 1 - PPMI + SVD", "",
              f"Vocabulary {t1['vocab_size']} lower-cased types from {t1['train_tokens']} train tokens; "
              f"window +/-{t1['config']['window']}, PPMI alpha={t1['config']['alpha']}, SVD {t1['config']['dim']}d.", "",
              "| Metric | Value |", "|---|---|",
              f"| WordSim-353 Spearman (covered pairs) | {w['spearman_covered']:.4f} ({w['n_covered']}/353 pairs) |",
              f"| WordSim-353 Spearman (all 353, OOV sim = 0) | {w['spearman_all353']:.4f} |",
              f"| Google analogy accuracy (all 19,544, OOV = wrong) | {a['acc_all']:.4f} |",
              f"| Google analogy accuracy (covered) | {a['acc_covered']:.4f} ({a['n_covered']} questions) |",
              f"| MST UUAS vs gold, 20 test sentences | {s['uuas']:.4f} ({s['correct']}/{s['gold_edges']} edges) |",
              f"| - linear-chain baseline / random-tree baseline | {b['linear_chain_uuas']:.4f} / {b['random_tree_uuas']:.4f} |",
              "", "Ablation (PPMI smoothing alpha x SVD weighting power):", "",
              _md(pd.read_csv(out / "task1" / "ablation.csv")), "", "![t1](task1/task1_ablation.png)", ""]
    if (out / "task2" / "results.json").is_file():
        t2 = load_json(out / "task2" / "results.json")
        summ = pd.read_csv(out / "task2" / "layer_summary.csv")
        r += ["## Task 2 - Structural probe on bert-base-uncased", "",
              f"Best layer **l\\* = {t2['best_layer']}** (selected on {t2['selected_on']} UUAS). "
              f"Test-argmax layers: UUAS {t2['best_layer_by_test_uuas']}, depth Spearman "
              f"{t2['best_layer_by_test_depth_spearman']}.", "",
              "Per layer (max over ranks):", "", _md(summ), "",
              "![t2](task2/task2_uuas_spearman_vs_layer.png)", "",
              "All runs: `task2/probe_results.csv`.", ""]
    if (out / "task3" / "results.json").is_file():
        t3 = load_json(out / "task3" / "results.json")
        sig = t3["significance"]
        r += ["## Task 3 - BERT-derived static vectors vs PPMI+SVD", "",
              _md(pd.read_csv(out / "task3" / "comparison.csv")), "",
              f"- WordSim paired bootstrap: delta {sig['wordsim_bootstrap']['delta']:+.4f}, "
              f"95% CI [{sig['wordsim_bootstrap']['ci95'][0]:+.4f}, {sig['wordsim_bootstrap']['ci95'][1]:+.4f}]",
              f"- Analogy McNemar: only-PPMI-correct {sig['analogy_mcnemar']['only_task1_correct']}, "
              f"only-BERT-correct {sig['analogy_mcnemar']['only_task3_correct']}, "
              f"p = {sig['analogy_mcnemar']['p_exact_mcnemar']:.3g}",
              f"- Structural (per-sentence Wilcoxon): BERT better on {sig['structural_wilcoxon']['n_better']}, "
              f"worse on {sig['structural_wilcoxon']['n_worse']}, tied {sig['structural_wilcoxon']['n_tie']}; "
              f"p = {sig['structural_wilcoxon']['p_wilcoxon']:.3g}", "",
              "![t3](task3/task3_vs_task1.png)", "", "Layer sweep of pooled vectors:", "",
              _md(pd.read_csv(out / "task3" / "layer_sweep.csv")), "", "![sweep](task3/task3_layer_sweep.png)", "",
              "### Discussion (generated from the numbers above)", ""]
        r += [f"- {s}" for s in t3["verdict"]["statements"]]
        r += ["", "Interpretation guide: WordSim-353 and analogies test semantic/lexical structure; the MST check "
              "tests whether cosine geometry of type-level vectors mirrors syntactic attachment. Averaging over "
              "contexts removes the token-specific information the probe exploits, so a gain on one evaluation "
              "need not carry over to the other.", ""]
    path = out / "REPORT.md"
    path.write_text("\n".join(r), encoding="utf-8")
    LOG.info("Report -> %s", path)
    return path


## 4. Task 1 — PPMI + SVD baseline

In [ ]:
!python run_all.py --tasks 1

## 5. Task 2 — Structural probe (GPU)

In [ ]:
!python run_all.py --tasks 2

## 6. Task 3 — BERT-derived static vectors vs Task 1

In [ ]:
!python run_all.py --tasks 3

## 7. Results

In [ ]:
from IPython.display import Markdown, Image, display
import pandas as pd
display(Markdown(open('outputs/REPORT.md').read().replace('](', '](outputs/')))
for p in ['outputs/dataset/dataset_stats.png', 'outputs/task1/task1_ablation.png',
          'outputs/task2/task2_uuas_spearman_vs_layer.png', 'outputs/task3/task3_vs_task1.png',
          'outputs/task3/task3_layer_sweep.png']:
    display(Image(p))

In [ ]:
# 8. Download results + checkpoints (BERT feature cache excluded - it is ~2 GB and regenerable)
!zip -qr project5_results.zip outputs checkpoints config.yaml
try:
    from google.colab import files; files.download('project5_results.zip')
except ImportError:
    print('Not on Colab: project5_results.zip written to', os.getcwd())